In [ ]:
from pathlib import Path
from getpass import getpass
import base64
import time

import cv2
import pandas as pd
import requests


API_KEY = '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'

DATASET_FOLDER = Path("../ImageDataset")
OUTPUT_FOLDER = Path("../ModelResults")

OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

MODELS = {
    "gpt_5_5": "openai/gpt-5.5",
    "gemini_3_flash": "google/gemini-3-flash-preview",
    "gemini_3_1_flash_lite": "google/gemini-3.1-flash-lite",
    "qwen_3_7_plus": "qwen/qwen3.7-plus"
}

PROMPT = (
    "Identify the main subject of the image and describe it in one concise, factual paragraph of 2–3 sentences. "
    "Include distinctive colors, materials, shapes, context, and any readable text that would help a visual search engine find similar objects or scenes. "
    "Avoid decorative language, unsupported assumptions, and phrases such as 'The image shows.' "
    "Express uncertain identifications cautiously using words such as 'likely,' 'possibly,' or 'appears to be.'"
)

ACCEPTED_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".webp",
}


def encode_image(image_path):
    image = cv2.imread(str(image_path))

    if image is None:
        raise ValueError(f"Could not read image: {image_path}")

    height, width = image.shape[:2]
    max_dimension = 1600

    scale = min(
        max_dimension / width,
        max_dimension / height,
        1.0,
    )

    if scale < 1.0:
        new_width = round(width * scale)
        new_height = round(height * scale)

        image = cv2.resize(
            image,
            (new_width, new_height),
            interpolation=cv2.INTER_AREA,
        )

    success, image_buffer = cv2.imencode(
        ".jpg",
        image,
        [cv2.IMWRITE_JPEG_QUALITY, 90],
    )

    if not success:
        raise ValueError(f"Could not encode image: {image_path}")

    return base64.b64encode(image_buffer).decode("utf-8")

def describe_image(image_b64, model_name):
    payload = {
        "model": model_name,
        "messages": [
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": PROMPT,
                    },
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": (
                                "data:image/jpeg;base64,"
                                + image_b64
                            )
                        },
                    },
                ],
            }
        ],
        "temperature": 0,
        "max_tokens": 300,
    }

    response = requests.post(
        "https://openrouter.ai/api/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {API_KEY}",
            "Content-Type": "application/json",
        },
        json=payload,
        timeout=120,
    )

    response.raise_for_status()

    response_data = response.json()

    return response_data["choices"][0]["message"]["content"]

if not DATASET_FOLDER.exists():
    raise FileNotFoundError(
        f"Dataset folder was not found: "
        f"{DATASET_FOLDER.resolve()}"
    )

image_paths = sorted(
    path
    for path in DATASET_FOLDER.iterdir()
    if path.is_file()
    and path.suffix.lower() in ACCEPTED_EXTENSIONS
)

if not image_paths:
    raise ValueError(
        f"No supported images were found in "
        f"{DATASET_FOLDER.resolve()}"
    )

print(f"Found {len(image_paths)} images.")
print(f"Models to evaluate: {len(MODELS)}")

encoded_images = {}

for image_path in image_paths:
    try:
        encoded_images[image_path] = encode_image(image_path)
    except Exception as error:
        encoded_images[image_path] = error


for model_index, (model_label, model_name) in enumerate(
    MODELS.items(),
    start=1,
):
    output_file = OUTPUT_FOLDER / f"submission_{model_label}.csv"

    print()
    print("=" * 70)
    print(
        f"Model {model_index}/{len(MODELS)}: "
        f"{model_name}"
    )
    print(f"Output: {output_file}")
    print("=" * 70)

    results = []

    for image_index, image_path in enumerate(
        image_paths,
        start=1,
    ):
        print(
            f"[{image_index}/{len(image_paths)}] "
            f"Processing: {image_path.name}"
        )

        encoded_value = encoded_images[image_path]
        start_time = time.perf_counter()

        try:
            if isinstance(encoded_value, Exception):
                raise encoded_value

            description = describe_image(
                encoded_value,
                model_name,
            )

            status = "success"

        except Exception as error:
            description = f"ERROR: {error}"
            status = "error"

        elapsed_seconds = round(
            time.perf_counter() - start_time,
            2,
        )

        results.append(
            {
                "image_path": image_path.as_posix(),
                "model": model_name,
                "status": status,
                "response_time_seconds": elapsed_seconds,
                "ai_response": description,
            }
        )

        pd.DataFrame(results).to_csv(
            output_file,
            index=False,
            encoding="utf-8-sig",
        )

    successful_requests = sum(
        result["status"] == "success"
        for result in results
    )

    print(
        f"Completed {model_name}: "
        f"{successful_requests}/{len(results)} successful requests."
    )

print()
print("Evaluation completed.")
print(f"Results were saved in: {OUTPUT_FOLDER.resolve()}")